<img src="https://datasciencedegree.wisconsin.edu/wp-content/themes/data-gulp/images/logo.svg" width="300">

# Behaviour when applying to data frames

#### Understanding indices and axes in numpy and pandas

notebook by Prof. Silviana Amethyst

---

💡 This notebook demonstates why the axis arguments to numpy and pandas are what they are, and hopefully helps to make them less counterintuitive.  Axis arguments in numpy and pandas indicate a direction of slicing or movement.
* `axis='rows'` indicates the operation happens as-if a looping variable is striding over the rows.  Do something for each row.
* `axis='columns'` indicates the operation happens as-if a looping variable is striding over the columns.  Do something for each column.

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# make data and a frame
data = np.random.normal(0,1,(6,4)) # 6 rows, 4 columns
df = pd.DataFrame(data=data,columns=['a','b','c','d'])

## Many ways of doing the same thing

This notebook first establishes two sets of calls which are equivalent, then seeks to impart a way of thinking about axis values

### The following are equivalent:

* `np.sum(data,0)`
* `df.sum(axis=0)`
* `df.apply(sum, axis='rows')`

They will all compute the sum *across* rows -- so, the column-sums from another perspective. 

In [ ]:
np.sum(data,0)

In [ ]:
df.sum(axis=0)

In [ ]:
df.apply(sum, axis='rows')

### The following are equivalent:

* `np.sum(data,1)`
* `df.sum(axis=1)`
* `df.apply(sum, axis='columns')`

They will all compute the sum *across* columns -- so, the row-sums from another perspective.  

In [ ]:
np.sum(data,1)

In [ ]:
df.sum(axis=1)

In [ ]:
df.apply(sum, axis='columns')

### How to think about the axis value for pandas / numpy

#### Why `rows` goes with `axis=0`

One way to think about the value `0` that's used for `rows` is that the 0th position of a multiindex in numpy is the one that varies.  That is, the first value of the sum that's computed in `np.sum(data,0)` can be looped to compute as 

In [ ]:
fixed_col_ind = 3 # three arbitrarily, cuz it's not 0, and i wanted to stay away from 0 and 1
s = 0             # because summing, start at 0. This is not the same zero as the `axis` value.

for ii in range(data.shape[0]):  # axis=0 -- get the size of the data in the 0th dimension
    s = s+data[ii,fixed_col_ind] # axis=0 -- use the looping variable in the 0th position
    #          ^^ 
    #         the one that's changing is position 0 in the [] accessor operator getting values from `data`

# sanity check -- will fail if I lied to you. 
assert(s==np.sum(data,0)[fixed_col_ind]) # axis=0

See?  It's the 0th position in the index into `data` that's changing.  

#### Why `columns` goes with `axis=1`

Columns in a matrix or ndarray are the 1th position in the multiindex into a numpy array.  The following code computes a sum of a row, and the column-index is the one that varies.  The `axis='columns'` option makes the column-index the one that varies, and so `axis='columns'` is equivalent to `axis=1`.  

In [ ]:
fixed_row_ind = 4 # arbitrarily choosing row 4 to be fixed
s = 0             # because summing, start at 0. This is not the same zero as the `axis` value.

for jj in range(data.shape[1]):          # <--- the 1 here
    s = s+data[fixed_row_ind,jj]
    #          0             1           # <--- is the 1 here
    #                       
    # the varying index value is the 1th one -- corresponding to columns.  Hence, `axis='columns'`.

# sanity check -- will fail if I lied to you.
assert(s==np.sum(data,1)[fixed_row_ind]) # <-- is the 1 here

#### Apply with a lambda/function has the same behaviour -- there's nothing special about `sum`

When we use the `axis='columns'` argument, we're saying that we want to collapse the data structure in the direction of the columns -- that dimension will disappear, in a way.  

This little demo code scales each number in the `b` column by 3, then truncates the number into an integer.  The result, since we used `axis='columns'`, is a series object of size 6x1 -- we started with a 6x4, and the 1th dimension has collapsed -- because `axis='columns'` is equivalent to `axis=1`.

In [ ]:
df.apply(lambda r: int(3*r['b']),axis='columns')

### Conclusion

Think of the axis value for pandas and numpy:
* as the axis index that varies in the looping that's being done for you,
* *not* as the object you're looping over